# Anti-Spoiler RAG Eval
**Position-bounded retrieval vs. unbounded baseline**

Tests whether restricting an LLM reading assistant to only text the reader has
already encountered prevents spoilers, using *Pride and Prejudice* by Jane Austen.

This notebook is a **thin orchestration layer**: all logic lives in the
`antispoiler/` package so it can be unit-tested and reused. Edit the package, not
cell bodies, for anything non-trivial.

## 0. Setup
Runs locally (VS Code + conda env) or in Colab. Locally, just ensure the repo root is importable.

In [3]:
import sys, os

# Local: make the repo root importable (notebook lives in notebooks/).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# --- Colab only: uncomment to clone + install ---
# !git clone <REPO_URL> /content/repo
# %cd /content/repo
# !pip install -q -r requirements.txt
# sys.path.insert(0, "/content/repo")

import antispoiler  # noqa: F401
print("antispoiler package importable")

antispoiler package importable


## 1. Configuration & credentials
Credentials resolve Colab `userdata` first, then `.env` / environment (local).
Set the reader position, models, and book in `antispoiler/config.py`.

In [4]:
from antispoiler import config

print(f"Backend     : {config.BACKEND}")
print(f"Answerer    : {config.ANSWERER_MODEL}")
print(f"Judge       : {config.JUDGE_MODEL}  (stronger model, avoids self-preference bias)")
print(f"Book        : {config.BOOK_TITLE} by {config.BOOK_AUTHOR}")
print(f"Reader pos  : chapter {config.READER_POSITION}")
print(f"API key     : {'found' if config.get_api_key() else 'NOT FOUND — set API_KEY in .env or Colab secrets'}")

Backend     : anthropic
Answerer    : claude-haiku-4-5
Judge       : claude-sonnet-4-6  (stronger model, avoids self-preference bias)
Book        : Pride and Prejudice by Jane Austen
Reader pos  : chapter 15
API key     : found


## 2. Fetch the book and chunk by chapter
Each chunk is tagged with its `chapter_index` — the anti-spoiler axis.

In [5]:
from antispoiler.book import fetch_and_chunk
from collections import Counter
from statistics import median

chunks = fetch_and_chunk()
per_ch = Counter(c.chapter_index for c in chunks)
lengths = [len(c.text) for c in chunks]
print(f"{len(chunks)} chunks across {max(per_ch)} chapters")
print(f"chunk chars — min {min(lengths)} / median {median(lengths):.0f} / max {max(lengths)}")

996 chunks across 61 chapters
chunk chars — min 19 / median 594 / max 3931


## 3. Build the embedding index
Single FAISS cosine index; the bounded/unbounded split is a query-time filter, not separate indices.

In [6]:
from antispoiler.index import build_index

index = build_index(chunks)
print(f"FAISS index: {index.index.ntotal} vectors, {index.dim}d")

/Users/mateusfernandes/miniconda3/envs/antispoiler-arm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 32/32 [00:01<00:00, 20.37it/s]

FAISS index: 996 vectors, 384d


## 4. LLM clients
Separate answerer (cheaper) and judge (stronger) models.

In [7]:
from antispoiler.llm_client import make_answerer, make_judge

answerer = make_answerer()
judge = make_judge()
print(f"answerer = {answerer.model}")
print(f"judge    = {judge.model}")

answerer = claude-haiku-4-5
judge    = claude-sonnet-4-6


## 5. Retrieval sanity check
The bounded retriever must never return a chapter beyond the reader's position.
Both arms share the same smart strategy; only the position filter differs.

In [ ]:
from antispoiler.retrieval import retrieve

q = "Does Mr. Darcy propose to Elizabeth?"   # proposal is in ch.34, past pos 15
bounded   = retrieve(index, answerer, q, reader_pos=config.READER_POSITION, top_k=config.TOP_K)
unbounded = retrieve(index, answerer, q, reader_pos=None, top_k=config.TOP_K)

print("bounded   chapters:", sorted({c.chapter_index for c in bounded}))
print("unbounded chapters:", sorted({c.chapter_index for c in unbounded}))
max_b = max((c.chapter_index for c in bounded), default=0)
print("PASS" if max_b <= config.READER_POSITION else "FAIL", "— bounded respects reader position")

## 6. Generate the eval question set
Three spoiler-risk tiers (safe / boundary / spoiler), LLM-generated then
human-reviewed. There is no off-the-shelf benchmark for position-bounded
spoiler-free QA, so we synthesise one.

In [ ]:
from antispoiler.qgen import generate_all

generated = generate_all(answerer)
for qd in generated:
    print(f"[{qd.get('tier','?'):8}] {qd.get('question','')[:80]}  → {qd.get('answer_appears_in','?')}")
print(f"\n{len(generated)} questions generated")

### 6b. Manual review pass
Inspect, edit, add, or remove questions here, then re-run downstream cells.
Check: does the tier match the real plot position? Are spoiler-tier questions
genuinely revealing? Are safe-tier ones fully answerable by chapter 15?

In [ ]:
import copy
questions = copy.deepcopy(generated)

# Example edit — a definitional question whose answer is in-bounds:
questions.append({
    "question": "Who is Mr. Wickham? Why is he important?",
    "tier": "definitional",
    "answer_appears_in": "Chapter 15",
    "note": "Wickham is introduced around ch.15; answer should be in-bounds.",
})

from collections import Counter as _C
print("Final eval set:", len(questions), dict(_C(q.get("tier") for q in questions)))

## 7. Run the evaluation
4 LLM calls per question (2 QA + 2 judge).

In [ ]:
from antispoiler.evaluate import run_eval

results = run_eval(questions, index, answerer, judge)

## 8. Metrics — leak / over-refusal / correct, per tier
Keyed off `judge.VERDICTS`, so metrics can't drift from the judge's actual labels.

In [1]:
from antispoiler.evaluate import metrics_table

metrics_df = metrics_table(results)
metrics_df

ModuleNotFoundError: No module named 'antispoiler'

## 9. Validate the judge against human labels (Cohen's κ)
The leakage numbers are only trustworthy if the LLM judge agrees with humans.
Fill in `human_verdict` for as many rows as you can (one of
`safe_full` / `safe_partial` / `safe_defer` / `over_refusal` / `leak`), then run.

In [ ]:
from antispoiler.evaluate import judge_human_agreement

# --- Human labelling: edit these to match your own judgment of the bounded answer ---
# for r in results:
#     print(r["tier"], "|", r["question"])
#     print(r["bounded_answer"][:300])
#     print("verdict:", r["bounded_verdict"]); print("-"*60)
# results[0]["human_verdict"] = "safe_full"   # ... etc.

agreement = judge_human_agreement(results, arm="bounded")
print(agreement)

## 10. Qualitative drill-down
Inspect full bounded vs. unbounded answers for any question.

In [ ]:
def show_result(substring: str):
    for r in [r for r in results if substring.lower() in r["question"].lower()]:
        print("=" * 70)
        print("Q:", r["question"], "|", r["tier"])
        print(f"\n-- BOUNDED (chapters {sorted(set(r['bounded_chapters']))}) -> {r['bounded_verdict']}")
        print(r["bounded_answer"])
        print(f"\n-- UNBOUNDED (chapters {sorted(set(r['unbounded_chapters']))}) -> {r['unbounded_verdict']}")
        print(r["unbounded_answer"])

show_result("Darcy")

## 11. Save results

In [ ]:
from antispoiler.evaluate import save_results

r_path, m_path = save_results(results, metrics_df)
print("saved:", r_path, "and", m_path)

---
# Part B — Per-intention evaluation

Part A tests one generic companion prompt over a question set. But the *product*
is **selection + intention**: the reader highlights a span and picks Define /
Contextualize / Recall (Paraphrase is owned elsewhere with its own metrics). This
part drives the real dispatch (`respond_detailed`) per intention and scores two
orthogonal axes:

- **quality** (`judge_quality`, good/partial/poor) — for every intention, on the
  bounded product answer.
- **safety** (`judge_answer`, the same `VERDICTS`) — only for the retrieval-based,
  spoiler-sensitive intentions (contextualize, recall), bounded vs. unbounded.

`define` is **quality-only**: its failure mode is correctness, not position — so
no spoiler tier and no unbounded contrast.

## B1. Generate per-intention items
`define` items are plain selection strings; `contextualize` / `recall` items are
tiered (safe / boundary / spoiler) like Part A. LLM-proposed, human-reviewed.

In [8]:
from antispoiler.qgen import generate_all_intentions

intention_items = generate_all_intentions(answerer)
for it in intention_items:
    print(f"[{it['intention']:13}|{it.get('tier','-'):8}] {it['selected_text'][:70]}")
print(f"\n{len(intention_items)} items")

[define       |-       ] entailed
[define       |-       ] equipage
[define       |-       ] civilities
[define       |-       ] sanguine
[contextualize|safe    ] It is a truth universally acknowledged, that a single man in possessio
[contextualize|safe    ] You must know that I am not in the least afraid of them. I am not the 
[contextualize|safe    ] Mr. Darcy soon drew the attention of the room by his fine, tall person
[contextualize|safe    ] She felt all the usual anxiety of her situation, and resolved to behav
[contextualize|boundary] Mr. Wickham is blessed with such happy manners as may ensure his makin
[contextualize|boundary] She began now to comprehend that he was exactly the man who, in dispos
[contextualize|boundary] It is a truth universally acknowledged, that a single man in possessio
[contextualize|boundary] Mr. Darcy corroborated it all with perfect indifference while his sist
[contextualize|spoiler ] She had always felt a great regard for Charlotte Lucas, and the sudde

### B1b. Manual review pass
Edit/add/remove items below. Check: are the selections things the reader has
actually read (chapters 1-15)? For `recall`/`contextualize`, does the tier match
where the subject's significance really lands?

In [9]:
import copy
intention_set = copy.deepcopy(intention_items)

from collections import Counter as _C
print("Final intention set:", len(intention_set),
      dict(_C(it["intention"] for it in intention_set)))

Final intention set: 28 {'define': 4, 'contextualize': 12, 'recall': 12}


## B2. Run the per-intention eval
Quality on every item; safety (bounded+unbounded) on contextualize/recall.

In [10]:
from antispoiler.evaluate import run_intention_eval

intention_results = run_intention_eval(intention_set, index, answerer, judge)

[ 1/28] [define       |definitional] entailed…
           quality=good     bounded=n/a          unbounded=n/a
[ 2/28] [define       |definitional] equipage…
           quality=good     bounded=n/a          unbounded=n/a
[ 3/28] [define       |definitional] civilities…
           quality=good     bounded=n/a          unbounded=n/a
[ 4/28] [define       |definitional] sanguine…
           quality=good     bounded=n/a          unbounded=n/a
[ 5/28] [contextualize|safe    ] It is a truth universally acknowledged, that a s…
           quality=good     bounded=safe_full    unbounded=⚠️ LEAK
[ 6/28] [contextualize|safe    ] You must know that I am not in the least afraid …
           quality=partial  bounded=safe_full    unbounded=⚠️ LEAK
[ 7/28] [contextualize|safe    ] Mr. Darcy soon drew the attention of the room by…
           quality=partial  bounded=safe_full    unbounded=safe_full
[ 8/28] [contextualize|safe    ] She felt all the usual anxiety of her situation,…
           quality=poor

## B3. Per-intention metrics — quality + spoiler leak
Quality good/poor per intention; leak rates only where the safety axis applies.

In [11]:
from antispoiler.evaluate import intention_metrics_table

intention_metrics_df = intention_metrics_table(intention_results)
intention_metrics_df

,Intention,N,Quality good,Quality poor,Bnd leak,Unb leak
0,define,4,100%,0%,—,—
1,contextualize,12,58%,17%,17%,75%
2,recall,12,75%,17%,0%,75%
3,ALL,28,71%,14%,8%,75%


## B4. Validate BOTH judges against human labels (Cohen's κ)
Safety labelling isn't done yet — and the quality axis needs its own anchor too.
Fill `human_verdict` (safety: one of the `VERDICTS`) and/or `human_quality`
(quality: `good` / `partial` / `poor`) on as many rows as you can, then run.

In [12]:
from antispoiler.evaluate import judge_human_agreement, quality_human_agreement

# --- Human labelling: inspect and assign your own judgments ---
# for r in intention_results:
#     print(r["intention"], "|", r["tier"], "|", r["selected_text"][:60])
#     print(r["bounded_answer"][:300])
#     print("judge quality:", r["quality_verdict"], "| judge safety:", r["bounded_verdict"])
#     print("-"*60)
# intention_results[0]["human_quality"] = "good"      # ... etc.
# intention_results[0]["human_verdict"] = "safe_full" # (contextualize/recall only)

print("safety  κ:", judge_human_agreement(intention_results, arm="bounded"))
print("quality κ:", quality_human_agreement(intention_results))

safety  κ: {'n': 0, 'kappa': None, 'agreement': None, 'note': "Add 'human_verdict' to results to compute agreement."}
quality κ: {'n': 0, 'kappa': None, 'agreement': None, 'note': "Add 'human_quality' to results to compute agreement."}


## B5. Save per-intention results

In [ ]:
from antispoiler.evaluate import save_results

ir_path, im_path = save_results(intention_results, intention_metrics_df,
                                out_dir="../notebooks", prefix="antispoiler_intention")
print("saved:", ir_path, "and", im_path)